In [1380]:
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, SpectralClustering
from sklearn.preprocessing import Binarizer, RobustScaler
from sklearn.metrics.pairwise import cosine_distances
import matplotlib.pyplot as plt
import numpy as np

# load data

In [1381]:
df = pd.read_csv('/Users/connorhall/datasets/inst414/module 3 assignment/Public.csv',
                 usecols=['SPECIES', 'AIRPORT', 'STATE', 'INCIDENT_DATE'])
df = df.dropna()

# remove unknown species
df = df[(~df['SPECIES'].str.contains('unknown|perching birds', case=False)) &
        (~df['AIRPORT'].str.contains('unknown', case=False))]
df

,INCIDENT_DATE,AIRPORT,STATE,SPECIES
8,1992-05-03,NORMAN Y. MINETA SAN JOSE INTL ARPT,CA,American robin
11,1995-04-14,ALEXANDRIA INTL,LA,Blackbirds
13,1994-09-01,SYRACUSE HANCOCK INTL,NY,Gulls
14,1990-09-17,OAKLAND COUNTY INTL,MI,Mourning dove
15,1990-07-13,LOS ANGELES INTL,CA,American kestrel
...,...,...,...,...
339266,2025-10-02,EVANSVILLE REGIONAL,IN,Chimney swift
339267,2025-10-01,SEATTLE-TACOMA INTL,WA,Cedar waxwing
339268,2025-10-02,CHICAGO O'HARE INTL ARPT,IL,Yellow-rumped warbler
339269,2025-10-03,ROANOKE REGNL ARPT/WOODRUM FIELD,VA,European starling


### filter data

In [1382]:
# remove low collision count for states and species
df = df[df.groupby('STATE')['STATE'].transform('count') >= 300]
df = df[df.groupby('SPECIES')['SPECIES'].transform('count') >= 100]

# species must appear in multiple places- avoids exotic species outliers (e.g. introduced species in Hawaii)
df = df[df.groupby('SPECIES')['STATE'].transform('nunique') >= 5]

df = df[['STATE', 'SPECIES']]
df

,STATE,SPECIES
8,CA,American robin
11,LA,Blackbirds
13,NY,Gulls
14,MI,Mourning dove
15,CA,American kestrel
...,...,...
339266,IN,Chimney swift
339267,WA,Cedar waxwing
339268,IL,Yellow-rumped warbler
339269,VA,European starling


# calculate collision counts

In [1383]:
counts = pd.crosstab(df['SPECIES'], df['STATE']).stack('STATE')
counts

SPECIES                STATE
American barn owl      AK         0
                       AL         4
                       AR         7
                       AZ         8
                       CA       783
                               ... 
Yellow-rumped warbler  VA        10
                       VT         3
                       WA        15
                       WI         5
                       WV         0
Length: 8160, dtype: int64

### separate airport and species index

In [1384]:
species = counts.index.get_level_values('SPECIES').tolist()
airports = counts.index.get_level_values('STATE').tolist()

counts_df = pd.DataFrame({'Species': species, 'Airport': airports})
counts_df['Count'] = counts.values
counts_df

,Species,Airport,Count
0,American barn owl,AK,0
1,American barn owl,AL,4
2,American barn owl,AR,7
3,American barn owl,AZ,8
4,American barn owl,CA,783
...,...,...,...
8155,Yellow-rumped warbler,VA,10
8156,Yellow-rumped warbler,VT,3
8157,Yellow-rumped warbler,WA,15
8158,Yellow-rumped warbler,WI,5


### add collision counts to new dataframe format

In [1385]:
# create lists of unique species names and airport names
unique_sp = sorted(list(set(species)))
unique_air = sorted(list(set(airports)))

# create list of collision count lists
counts_list = []
for sp in unique_sp:
    sp_filter = counts_df[counts_df['Species'] == sp]
    counts_list.append(sp_filter['Count'].tolist())
    
# create new dataframe format
collisions_df = pd.DataFrame(counts_list, columns=unique_air, index=unique_sp)
collisions_df

,AK,AL,AR,AZ,CA,CO,CT,DC,FL,FN,...,SC,SD,TN,TX,UT,VA,VT,WA,WI,WV
American barn owl,0,4,7,8,783,35,0,14,40,4,...,0,1,66,116,102,5,0,151,0,0
American coot,0,0,2,29,95,8,0,0,12,0,...,1,3,9,23,49,3,0,19,3,1
American crow,2,5,2,1,96,9,7,8,15,2,...,4,1,25,56,4,15,14,68,9,29
American golden-plover,18,1,1,0,1,0,6,7,3,0,...,1,0,1,36,0,1,3,1,18,0
American goldfinch,0,1,0,0,9,2,4,1,2,0,...,0,2,2,3,1,1,0,12,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Woodchuck,0,0,0,0,0,0,2,1,0,0,...,2,0,5,0,0,8,0,0,0,1
Yellow warbler,0,0,1,1,26,3,0,0,6,0,...,0,0,6,5,2,0,0,5,1,0
Yellow-bellied sapsucker,0,1,0,0,0,0,2,2,4,0,...,0,0,1,1,0,2,0,0,3,0
Yellow-crowned night heron,0,1,0,0,1,0,0,0,88,1,...,0,0,1,27,0,0,0,0,0,0


### verify output

In [1386]:
# compare sum of collisions from counts_df and collisions_df
counts_1 = counts_df.groupby('Species')['Count'].sum().values.tolist()
counts_2 = collisions_df.sum(axis=1).values.tolist()
counts_1 == counts_2

True

# clustering

In [ ]:
scaler = RobustScaler()
scaled_data = scaler.fit_transform(collisions_df)

# n_clusters=8 results in 0 dissimilarities below
model = AgglomerativeClustering(n_clusters=8, metric='cosine', linkage='complete')
labels = model.fit_predict(scaled_data)

pd.DataFrame(labels).value_counts()

0
3    34
7    29
4    28
1    24
2    19
0    14
6     6
5     6
Name: count, dtype: int64

### find clusters containing dissimilar rows

In [1388]:
copy_df = collisions_df.copy()
copy_df['label'] = labels

In [1389]:
for label in range(len(np.unique(labels))):
    row_count = 0
    dissimilar_count = 0

    for sp in copy_df[copy_df['label'] == label].index.tolist():
        row_count += 1
        airport_list1 = counts_df[(counts_df['Species'] == sp) & (counts_df['Count'] > 0)]['Airport'].values.tolist()
        
        for sp2 in copy_df[copy_df['label'] == label].index.tolist():
            if sp2 == sp:
                continue
            
            airport_list2 = counts_df[(counts_df['Species'] == sp2) & (counts_df['Count'] > 0)]['Airport'].values.tolist()
            # if neither species appears in the same state
            if len(set(airport_list1) & set(airport_list2)) == 0:
                dissimilar_count+=1
                
    print(f'cluster cat: {label}, dissimilarities: {dissimilar_count}, total rows: {row_count}')
    

cluster cat: 0, dissimilarities: 0, total rows: 14
cluster cat: 1, dissimilarities: 0, total rows: 24
cluster cat: 2, dissimilarities: 0, total rows: 19
cluster cat: 3, dissimilarities: 0, total rows: 34
cluster cat: 4, dissimilarities: 0, total rows: 28
cluster cat: 5, dissimilarities: 0, total rows: 6
cluster cat: 6, dissimilarities: 0, total rows: 6
cluster cat: 7, dissimilarities: 0, total rows: 29
